# 🗃️ 第 23 课 · KV 块分配器:free list、引用计数与碎片

> BlockAllocator 逐行实现 · 块池热力图 · 压力测试 · vLLM block_pool 对照 · 真实显存

**章节**: 第 4 章 · 模型执行器与 CUDA 优化

---

> **📚 本笔记本属于《VLLM_learn: 图解 vLLM 推理引擎》系列**
> 风格参照《鸢尾花书》: 重图解、重直觉、循序渐进、动手实操

## 🎯 学习目标

- 理解为什么 KV cache 必须按块管理,而不是整段预分配
- 手写 BlockAllocator:free list 分配 + 引用计数回收,逐行推演
- 玩一次典型场景:分配 4 条序列、释放中间一条,观察碎片形态
- 画块池热力图,看引用计数与空闲空洞
- 压力测试:几百步随机分配/释放,比较 free_list 与 best_fit 的碎片率
- 对照 vLLM block_pool.py 的三张账(空闲表 / 引用计数 / 分配器)
- 用真实 GPU 微基准算出「一块 KV 到底占多少显存」

## 📑 本课目录

1. **直觉:仓库里的储物柜** — 按块存取, 随用随取
2. **为什么按块管理 KV Cache** — 整段预分配 vs 按需分块的账本
3. **手写 BlockAllocator** — free list + refcount, 逐行推演
4. **典型场景** — 分配 4 条序列, 释放中间一条
5. **块池热力图** — 把 48 块画成一行格子
6. **压力测试** — free_list vs best_fit 的碎片率
7. **对应 vLLM:block_pool.py 的三张账** — 真实实现对照
8. **真实显存:一池 KV 块到底多大** — kv_bytes_real 实测

## 🔗 参考资料

- [PagedAttention (SOSP'23)](https://arxiv.org/abs/2309.06180)
- [vLLM block_pool 源码](https://github.com/vllm-project/vllm/blob/main/vllm/v1/core/block_pool.py)
- [vLLM 官方博客](https://blog.vllm.ai/2023/06/20/vllm.html)

## 1. 直觉:仓库里的储物柜 🗃️

想象一间**存放柜的仓库**:柜子分成一个个小格子,客人来了**按需租格子**,退租时格子归还。
好处是显而易见的:

- 客人只租自己需要的大小,不浪费;
- 格子可以分散在仓库各处,不必连成一片;
- 两个客人可以**共享**同一排格子(前缀复用)。

KV cache 的显存管理完全一样:把显存切成**固定大小的块(block)**,每个请求按需占块。
这就是 PagedAttention(arXiv:2309.06180)的核心思想——把「整段连续显存」变成「可分的页」。

本课实现一个**最小可运行的块分配器**。

## 2. 为什么按块管理 KV Cache 📦

**传统做法(整段预分配)**:每条序列一进来,就按它的最大长度 `max_len` 预留一整条 KV 显存。
问题:
1. 大部分请求生成不到 `max_len`,预留的显存**白白锁死**;
2. 序列释放后留下**无法复用的空洞**(碎片);
3. 无法跨请求共享(前缀缓存不可行)。

**按块管理(分页)**:显存切成固定大小块(如每块 16 token),按需分配、用完归还。

$$ \text{需要的块数} = \left\lceil \frac{\text{seq\_len}}{\text{block\_size}} \right\rceil $$

最后一块装不满是「块内尾部浪费」,通常只有 block_size/2 ≈ 8 个词元,远小于整段预分配的浪费。vLLM 博客报告:PagedAttention 的内存浪费**低于 4%**。

## 3. 手写 BlockAllocator ✍️

核心逻辑只有四条:
1. **free list**:空闲块号队列,分配时取队首;
2. **引用计数**:每块记录被几个序列引用,`refcount>0` 说明有人用;
3. **释放**:`refcount` 减一,**减到 0 才回收进 free list**;
4. **共享**:同一块可被多个序列引用(前缀复用),释放时只减计数。

我们实现两种找块策略:`free_list`(直接取队首,快)与 `best_fit`(扫全池找最小连续空洞,省外部碎片)。

✍️ **注意 `release` 里「减到 0 才回收」**:这正是共享块(refcount>1)能安全存在的关键——
一个人退租,室友还在住,柜子不能下架。

In [ ]:
# -*- coding: utf-8 -*-
import numpy as np                                 # 数值库
from dataclasses import dataclass, field          # 数据类

class BlockAllocator:
    """KV 块分配器: free list + 引用计数。

    num_blocks  : 物理块总数
    block_size  : 每块容纳的 token 数
    strategy    : 'free_list' (队首取块) / 'best_fit' (扫池找最小空洞)
    """

    def __init__(self, num_blocks=48, block_size=16, strategy="free_list"):
        self.num_blocks = num_blocks               # 物理块总数
        self.block_size = block_size               # 每块 token 数
        self.strategy = strategy                   # 分配策略
        # free list: 初始全部空闲 (0..num_blocks-1)
        self.free_list = list(range(num_blocks))   # 空闲块号队列
        self.refcounts = [0] * num_blocks          # 每块引用计数
        self.owners = {}                           # owner_id -> [block_ids] 谁占了哪些块

    def allocate(self, num_tokens, owner_id):
        """为 owner 分配 num_tokens 所需的块。返回块号列表; 失败返回 None。"""
        need = (num_tokens + self.block_size - 1) // self.block_size  # 需要的块数 = ceil
        if self.strategy == "free_list":           # free list: 直接取队首 need 块
            blocks = self.free_list[:need]         # 队首切片
            if len(blocks) < need:                 # 不够
                return None                        # 分配失败
            self.free_list = self.free_list[need:] # 从队首移除
        else:                                      # best_fit: 扫全池找最小连续空洞
            # 收集所有「连续空闲段」 (起点, 长度)
            free_runs = []                         # 空闲段列表
            run_start, run_len = None, 0           # 当前段起点 / 长度
            for b in range(self.num_blocks):       # 遍历全池
                if self.refcounts[b] == 0:         # 空闲块
                    if run_len == 0:               # 新段起点
                        run_start = b
                    run_len += 1                   # 段长 +1
                else:                              # 被占用, 段中断
                    if run_len > 0:                # 有段要收
                        free_runs.append((run_start, run_len))
                    run_len = 0                    # 重置
            if run_len > 0:                        # 收尾段
                free_runs.append((run_start, run_len))
            free_runs.sort(key=lambda x: x[1])     # 按段长升序 (最小空洞优先)
            blocks = []                            # 选中的块
            for start, ln in free_runs:            # 从小到大填洞
                for b in range(start, start + min(ln, need - len(blocks))):
                    blocks.append(b)               # 取块
                if len(blocks) >= need:            # 够了
                    break
            if len(blocks) < need:                 # 全池仍不够
                return None                        # 分配失败
        # 更新引用计数 + 记录 owner
        for b in blocks:                           # 每块
            self.refcounts[b] += 1                 # 引用 +1
        self.owners[owner_id] = blocks             # 记录 owner -> 块
        return blocks                              # 返回块号列表

    def release(self, owner_id):
        """释放 owner 的所有块: 每块 refcount-1, 减到 0 才回收进 free list。"""
        blocks = self.owners.pop(owner_id, [])     # 取出该 owner 的块并删除记录
        for b in blocks:                           # 每块
            self.refcounts[b] -= 1                 # 引用 -1
            if self.refcounts[b] == 0:             # 没人用了
                self.free_list.append(b)           # 回收进 free list

    def share(self, src_owner, dst_owner, n_blocks):
        """共享: 把 src 的前 n_blocks 块也让 dst 引用 (前缀复用)。"""
        blocks = self.owners[src_owner][:n_blocks] # 源的前 n 块
        for b in blocks:                           # 每块
            self.refcounts[b] += 1                 # 引用 +1 (共享!)
        self.owners[dst_owner] = list(blocks)      # dst 拥有这些块


# --------------------------------------------------------------------------
# 演练: 小分配器, 演示 refcount 语义
# --------------------------------------------------------------------------
a = BlockAllocator(num_blocks=8, block_size=4)     # 8 块, 每块 4 token
b1 = a.allocate(5, "seq_1")                        # 5 token -> ceil(5/4)=2 块
b2 = a.allocate(9, "seq_2")                        # 9 token -> 3 块
print(f"seq_1 占块 {b1}, refcounts = {a.refcounts}")
print(f"seq_2 占块 {b2}, refcounts = {a.refcounts}")
a.release("seq_1")                                 # 释放 seq_1
print(f"释放 seq_1 后: refcounts = {a.refcounts}")
print(f"  free_list = {a.free_list}  <- 释放的块回到队首/队尾待复用")

## 4. 玩一次典型场景 🎬

分配 4 条序列(64/96/32/128 词元,block_size=16 → 4/6/2/8 块),再释放中间的 `seq_1`,
观察 free list 与 refcounts 的变化。

🎬 **释放 seq_1 后,中间挖出一条 6 块的「洞」**——它和尾部的空闲块不相连,这就是最直观的碎片形态。

In [ ]:
# -*- coding: utf-8 -*-
alloc = BlockAllocator(num_blocks=48, block_size=16, strategy="free_list")  # 48 块, 每块 16 token
seqs = {"seq_0": 64, "seq_1": 96, "seq_2": 32, "seq_3": 128}   # 4 条序列的 token 数
for sid, n_tok in seqs.items():                  # 逐条分配
    blks = alloc.allocate(n_tok, sid)            # 分配
    print(f"{sid}: {n_tok:4d} token -> {len(blks)} 块 {blks}")

print("\n分配后 refcounts (前 24 块):", alloc.refcounts[:24])
alloc.release("seq_1")                            # 释放中间的 seq_1
print("释放 seq_1 后 free_list 前 10 个:", alloc.free_list[:10])
print("释放 seq_1 后 refcounts (前 24 块):", alloc.refcounts[:24])

## 5. 可视化:块池热力图 🎨

把 48 个块画成一行格子:灰色 = 空闲(refcount=0),颜色越深 = 被越多序列引用。

🎨 **中间那一截浅色就是被释放的空洞**;深色块 refcount>1(被共享)。

In [ ]:
# -*- coding: utf-8 -*-
import matplotlib.pyplot as plt                   # 绘图库
%matplotlib inline
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

fig, ax = plt.subplots(figsize=(12, 1.6))          # 画布 (一行格子)
rc = np.array(alloc.refcounts).reshape(1, -1)      # (1, 48) 引用计数行
im = ax.imshow(rc, cmap="YlGnBu", aspect="auto")   # 热力图: 越深引用越多
ax.set_xticks(range(alloc.num_blocks))             # 每块一个刻度
ax.set_xticklabels(range(alloc.num_blocks), fontsize=6)  # 块号
ax.set_yticks([])                                  # 隐藏 y 轴
ax.set_title("块池热力图: 灰=空闲, 深蓝=被多序列引用; 中间浅色段 = 释放出的空洞")  # 标题
fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.05)   # 色条
plt.tight_layout()
plt.show()
print("refcounts:", alloc.refcounts)              # 打印具体引用计数

## 6. 压力测试:随机分配/释放序列 🌀

真实服务里,序列**源源不断地来、又随机地走**(生成完毕或被抢占)。
我们模拟几百步随机操作,比较 free_list 与 best_fit 两种策略的**外部碎片率**
(= 1 − 最大连续空闲段 / 总空闲块,衡量「空但零散」的块占比)。

注意两种策略在「块内浪费」(internal)上**完全相同**——都要 `ceil(n/block_size)` 块,
末尾装不满的那点由块大小决定,与选哪几块无关。真正能拉开差距的是**外部碎片**:
空闲块被切得多碎、还能不能拼出大段连续空间。这才是 best_fit 发挥作用的地方。

🌀 **看曲线**:best_fit 的平均与峰值外部碎片率都更低——它把分配压进最小的空洞,把大段连续空闲留给了后来的大请求。代价是每次分配要扫全池找洞,更慢。
工程上仍常用 free_list:分页下块可分散,外部碎片并不致命,而速度是硬要求。

In [ ]:
# -*- coding: utf-8 -*-
import numpy as np                                 # 数值库
import matplotlib.pyplot as plt                   # 绘图库
%matplotlib inline
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

def largest_free_run(a):
    """返回池里最大一段「连续空闲块」的长度 (外部碎片的关键指标)。"""
    best, run = 0, 0                              # 最大段 / 当前段
    for c in a.refcounts:                         # 遍历每块
        if c == 0:                                # 空闲块
            run += 1                              # 段长 +1
            best = max(best, run)                 # 更新最大值
        else:                                     # 被占用
            run = 0                               # 段中断
    return best                                   # 返回最大连续空闲段

def stress(strategy, steps=300, num_blocks=64, block_size=16, seed=0):
    """随机分配/释放压力测试。返回 外部碎片率 历史 (0=完全连续, 1=碎到最大段只占1块)。"""
    rng = np.random.default_rng(seed)              # 可复现随机数
    a = BlockAllocator(num_blocks=num_blocks, block_size=block_size, strategy=strategy)  # 分配器
    frag_hist = []                                 # 每步外部碎片率
    active = {}                                    # owner -> token 数
    next_id = 0                                    # owner 编号
    for step in range(steps):                      # 每一步
        if rng.random() < 0.6 or not active:       # 60% 概率分配
            n_tok = int(rng.integers(8, 200))      # 随机 token 数
            owner = f"r{next_id}"; next_id += 1    # 新 owner
            blks = a.allocate(n_tok, owner)        # 分配
            if blks is not None:                   # 成功
                active[owner] = n_tok              # 记录
        else:                                      # 否则释放
            owner = rng.choice(list(active))       # 随机选一个
            a.release(owner)                       # 释放
            del active[owner]                      # 删除记录
        used_blocks = sum(1 for c in a.refcounts if c > 0)  # 已占用块
        free_blocks = a.num_blocks - used_blocks  # 总空闲块
        if free_blocks > 0:                       # 有空闲
            frag = 1 - largest_free_run(a) / free_blocks  # 外部碎片率
        else:                                     # 全满
            frag = 0.0                            # 无空闲则无碎片
        frag_hist.append(frag)                     # 记录
    return frag_hist

hist_f = stress("free_list", seed=0)               # free_list 外部碎片历史
hist_b = stress("best_fit", seed=0)                # best_fit 外部碎片历史

fig, ax = plt.subplots(figsize=(9, 4))             # 画布
ax.plot(hist_f, color="#4C72B0", lw=1, label="free_list", alpha=0.8)   # free_list 曲线
ax.plot(hist_b, color="#C44E52", lw=1, label="best_fit", alpha=0.8)   # best_fit 曲线
ax.set_xlabel("随机操作步")                         # x 轴
ax.set_ylabel("外部碎片率")                         # y 轴
ax.set_title("压力测试: free_list vs best_fit 的外部碎片率")   # 标题
ax.legend()                                        # 图例
ax.grid(alpha=0.3)                                 # 网格
plt.tight_layout()
plt.show()

# 汇总: 平均与峰值外部碎片率
import numpy as np                                 # 数值库
print(f"free_list: 平均外部碎片率 = {np.mean(hist_f):.3f}, 峰值 = {np.max(hist_f):.3f}")
print(f"best_fit : 平均外部碎片率 = {np.mean(hist_b):.3f}, 峰值 = {np.max(hist_b):.3f}")
print(f"=> best_fit 优先填最小连续空洞, 保住大段连续空闲, 外部碎片率更低;"
      f"free_list 只取队首, 可能把大段切碎。")

## 7. 对应 vLLM:block_pool.py 的三张账 🔍

vLLM V1 的 `vllm/v1/core/block_pool.py` 管理的就是本课这套账(概念对照):

| 本课 | vLLM | 说明 |
|---|---|---|
| `free_list` | `free_block_queue`(双端队列) | 空闲物理块队列, 分配时出队 |
| `refcounts` | 每块的引用计数 | 共享块 (前缀缓存 / 并行采样) 计数 |
| `allocate` | `allocate_slots` | 调度器按 token 数申请块 |
| `release` | `free` / 归还 | refcount 归零才回池 |

vLLM 的 `GPUBlockPool` 启动时按 `gpu_memory_utilization` 一次划出成百上千块;每块默认 **16 token**。调度器每步 `allocate_slots` 给每个请求的 `num_new_tokens` 分配新块。

> 📄 引用:vLLM V1 博客(Anatomy of vLLM)描述:*「Each block stores 16 tokens by default.
> If a request has 17 new tokens, we need ceil(17/16)=2 blocks」*。

## 8. 真实显存:一池 KV 块到底多大 💾

上面分配器管的是「块号」(虚拟地块),没给「一块到底占多少显存」。真实账本:

$$ \text{一块字节} = 2 \times \text{layers} \times \text{kv\_heads} \times \text{head\_dim} \times \text{block\_size} \times \text{dtype\_bytes} $$

用 ch02 的 `real_ops.kv_bytes_real` 在真实 GPU 上量化。

💾 **真实显存数字**。KV 随序列长度线性膨胀——所以 vLLM 必须「按需分块」而不是一进场就整段锁死,否则池子早被大 prompt 撑爆。

In [ ]:
# -*- coding: utf-8 -*-
import sys, os                                   # 系统库
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # OpenMP 兼容
sys.path.insert(0, "..")
sys.path.insert(0, r"..\ch02")
from real_ops import kv_bytes_real                # ch02 真实 KV 显存微基准

# 真实配置: 16 层, 8 个 KV head (GQA), D=128, 一条 2048 词元的序列, bf16 KV
NL, KH, HD, SEQ = 16, 8, 128, 2048                # 层数 / KV头 / head_dim / 序列长度
r = kv_bytes_real(L=NL, kv_heads=KH, head_dim=HD, seq=SEQ, batch=1, dtype="bf16")
per_tok = 2 * NL * KH * HD * 2                    # 每词元 K+V 字节 (bf16)
print("设备:", r.get("device"))
print(f"理论 KV 字节 = 2 × {NL}层 × {KH}头 × {HD}维 × {SEQ}词元 × 2B = {r['theory_bytes'] / 1e6:.1f} MB")
print(f"GPU 实际分配  = {r['alloc_bytes'] / 1e6:.1f} MB (与理论一致, 双通道 K+V)")
print(f"单块 (block_size=16) 承载 = {16 * per_tok / 1e3:.1f} KB")
print(f"8GiB 显存约可同时放 {round(8 * 1024**3 / (r['theory_bytes'] + 1e-12))} 条这种 {SEQ} 序列 —— "
      f"这就是分配器要精打细算的总盘子")

## 9. 🖥️ Streamlit 动态演示:KV 块分配器模拟器

运行 `app_23_allocator.py`:可点击的分配器模拟器——手动分配/释放,实时看块池热力图。

### 📜 App 完整源码(`app_23_allocator.py` 嵌入)

▶️ 此 cell 在 streamlit 环境中才真正运行;在 notebook 中仅作展示。

In [ ]:
try:
    import streamlit as st
    _IS_STREAMLIT = bool(st.runtime.exists())
except Exception:
    _IS_STREAMLIT = False

if _IS_STREAMLIT:
    # -*- coding: utf-8 -*-
    """🗃️ app_23_allocator.py — KV 块分配器模拟器(VLLM_learn 第 4 章 · 第 23 课)

    运行: streamlit run app_23_allocator.py
    """
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go
    import plotly.express as px
    import streamlit as st

    st.set_page_config(page_title="🗃️ 23 · KV 块分配器模拟", layout="wide")
    st.title("🗃️ 第 23 课配套 App · KV 块分配器模拟器")
    st.markdown(
        "vLLM 用 **块分配器** 管理 KV Cache:空闲块组成 free list,每个块带 **refcount(引用计数)**,"
        "序列申请块、释放块,块可能因'中间空洞'产生 **外部碎片**。本 App 让你亲手分配/释放序列,"
        "实时观察块池状态与碎片率。"
    )

    if "alloc" not in st.session_state:
        st.session_state.alloc = None
    if "history" not in st.session_state:
        st.session_state.history = []


    class SimAllocator:
        def __init__(self, num_blocks, block_size, strategy="free_list"):
            self.num_blocks = num_blocks
            self.block_size = block_size
            self.strategy = strategy
            self.free = list(range(num_blocks))
            self.refcount = [0] * num_blocks
            self.seqs = {}

        def _pick(self, k):
            if self.strategy == "free_list":
                return self.free[:k]
            sizes = []
            run, start = 0, None
            used = set(b for b in range(self.num_blocks) if self.refcount[b] > 0)
            for b in range(self.num_blocks):
                if b not in used:
                    if run == 0:
                        start = b
                    run += 1
                else:
                    if run > 0:
                        sizes.append((run, start))
                    run = 0
            if run > 0:
                sizes.append((run, start))
            sizes.sort()
            got = []
            for r, s in sizes:
                for b in range(s, s + min(r, k - len(got))):
                    got.append(b)
                if len(got) >= k:
                    break
            return got

        def allocate(self, seq_id, n_tokens):
            k = int(np.ceil(n_tokens / self.block_size))
            if k > len(self.free):
                return False
            blocks = self._pick(k)
            for b in blocks:
                self.free.remove(b)
                self.refcount[b] += 1
            self.seqs[seq_id] = blocks
            return True

        def release(self, seq_id):
            if seq_id not in self.seqs:
                return False
            for b in self.seqs.pop(seq_id):
                self.refcount[b] -= 1
                if self.refcount[b] == 0:
                    self.free.append(b)
                    self.free.sort()
            return True

        def stats(self):
            used = sum(1 for r in self.refcount if r > 0)
            free_blocks = self.num_blocks - used
            runs, run, start = [], 0, None
            for b in range(self.num_blocks):
                if self.refcount[b] == 0:
                    if run == 0:
                        start = b
                    run += 1
                else:
                    if run > 0:
                        runs.append((start, run))
                    run = 0
            if run > 0:
                runs.append((start, run))
            largest = max((r for _, r in runs), default=0)
            frag = 1.0 - (largest / free_blocks) if free_blocks > 0 else 0.0
            return dict(used=used, free=free_blocks, runs=runs, largest_run=largest,
                        fragmentation=frag)


    with st.sidebar:
        st.header("🎛️ 池配置")
        num_blocks = st.slider("块总数", 16, 256, 48)
        block_size = st.slider("块大小 block_size(词元/块)", 4, 64, 16, step=4)
        strategy = st.radio("分配策略", ["free_list(顺序取最小块号)", "best_fit(优先填小空洞)"])
        st.header("🕹️ 操作")
        seq_tokens = st.number_input("新序列长度(词元)", 1, 4096, 64, step=8)
        if st.button("➕ 分配一个新序列", width="stretch"):
            a = SimAllocator(num_blocks, block_size, "free_list" if "free_list" in strategy else "best_fit")
            st.session_state.alloc = a
            st.session_state.history = []
            sid = f"seq_{len(st.session_state.alloc.seqs)}"
            ok = st.session_state.alloc.allocate(sid, int(seq_tokens))
            st.session_state.history.append(("allocate", sid, int(seq_tokens), ok))
        if st.button("🔄 重置并跑一个典型场景", width="stretch"):
            a = SimAllocator(num_blocks, block_size, "free_list" if "free_list" in strategy else "best_fit")
            st.session_state.alloc = a
            st.session_state.history = []
            for i, ln in enumerate([64, 96, 32, 128]):
                ok = a.allocate(f"seq_{i}", ln)
                st.session_state.history.append(("allocate", f"seq_{i}", ln, ok))
            st.session_state.alloc.release("seq_1")
            st.session_state.history.append(("release", "seq_1", 96, True))

    alloc = st.session_state.alloc
    if alloc is None:
        alloc = SimAllocator(num_blocks, block_size, "free_list")
        st.session_state.alloc = alloc

    s = alloc.stats()
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("已用块 / 总块", f"{s['used']} / {num_blocks}")
    c2.metric("空闲块", f"{s['free']}")
    c3.metric("最大连续空闲段", f"{s['largest_run']} 块")
    c4.metric("外部碎片率", f"{s['fragmentation'] * 100:.1f}%",
              help="碎片率 = 1 − 最大连续空闲段 / 空闲块数,越接近 0 越好")

    st.markdown("### 🗂️ 块池状态(颜色 = 引用计数)")
    st.caption("每个格子是一个 KV 块。灰 = 空闲;颜色越深 = 被越多序列共享引用(refcount 越大)。")
    colors = [alloc.refcount[b] for b in range(num_blocks)]
    fig_grid = go.Figure(go.Heatmap(
        z=[colors], colorscale=[[0, "#e0e0e0"], [0.4, "#7fb3d5"], [1, "#1b4f72"]],
        zmin=0, zmax=max(4, max(colors)),
        x=[f"块{b}" for b in range(num_blocks)], y=["池"],
        hovertemplate="块 %{x}<br>refcount=%{z}<extra></extra>",
    ))
    fig_grid.update_layout(height=140, xaxis_tickangle=-60, yaxis_visible=False)
    st.plotly_chart(fig_grid, width="stretch")

    st.markdown("### 📈 空闲段分布")
    st.caption("空闲块按连续段展示:若碎片率 > 0,说明存在'可用但拼不成大段'的块,大序列可能因找不到连续段而分配失败。")
    runs = s["runs"]
    bar_df = pd.DataFrame({
        "空闲段": [f"段{i}(块{r[0]}~{r[0]+r[1]-1})" for i, r in enumerate(runs)] or ["(无)"],
        "块数": [r[1] for r in runs] or [0],
    })
    fig_runs = px.bar(bar_df, x="空闲段", y="块数", text="块数")
    fig_runs.update_layout(height=300)
    st.plotly_chart(fig_runs, width="stretch")

    st.markdown("### 📋 当前序列的块分配")
    st.caption("每序列占用的块号列表。块可以跨序列共享(refcount > 1),这是 vLLM prefix caching 的基础。")
    rows = [{"序列": sid, "长度(词元)": len(b) * alloc.block_size,
             "块数": len(b), "块号": str(b)} for sid, b in alloc.seqs.items()]
    if rows:
        st.dataframe(pd.DataFrame(rows), width="stretch", hide_index=True)
    else:
        st.info("池里还没有任何序列,点左侧「➕ 分配一个新序列」试试。")

    st.markdown("---")
    st.markdown(
        "💡 **直觉**:KV Cache 像一间存放柜的仓库 🗃️。free list 是「空柜清单」,refcount 是「一把钥匙几个人在用」。"
        "碎片率衡量的是仓库里「空但零散」的柜子占比 —— 这正是 PagedAttention 相比整段预分配要解决的经典难题。"
    )
    st.caption("《VLLM_learn: 图解 vLLM 推理引擎》第 4 章 · 第 23 课配套演示")

    # ============ PyCharm / 直接运行入口 ============
    # 说明: 在 PyCharm 里直接 Run 本文件,即可启动 Streamlit 服务(浏览器打开 http://localhost:8501)。
    # 与命令行 `streamlit run app_XX.py` 完全等价(用子进程方式, 避免与顶层 st 调用冲突)。
    if __name__ == "__main__":
        # 如果已在 Streamlit Runtime 中(AppTest/嵌入时加载),不再启动子进程。
        try:
            import streamlit.runtime as st_runtime
            if st_runtime.exists():
                raise SystemExit(0)
        except Exception:
            pass
        import os
        import subprocess
        import sys
        subprocess.run([sys.executable, "-m", "streamlit", "run", os.path.abspath(__file__)])

else:
    print("💡 当前不是 streamlit 环境, 跳过执行本 App。")
    print("    请直接运行: python -m streamlit run app_23_allocator.py")


### 🏃 运行方法

```
python -m streamlit run app_23_allocator.py
```
浏览器打开 http://localhost:8501 ,点击分配/释放,观察块池热力图与碎片率。

## ✅ 本课小结

- 按块管理 = 分页思想:显存切成固定块,按需分配、用完归还,浪费低于 4%
- BlockAllocator 四要素:free list / 引用计数 / 释放(减到0才回收) / 共享(前缀复用)
- 典型场景展示碎片形态:释放中间序列挖出与尾部不相连的空洞
- 压力测试:块内浪费(internal)两种策略相同;best_fit 优先填小空洞降低外部碎片,但分配要扫全池,free_list 快而碎片略高
- vLLM 对照:free_block_queue / refcount / allocate_slots / free,每块默认 16 token
- 真实显存:一块 16 token 的 KV 占 ~数百 KB~MB,整池是分配器精打细算的总盘子

## ✍️ 动手练习

1. 给 BlockAllocator 加一个「空闲块数不足时先压缩空洞」的逻辑,对比碎片率
2. 实现共享的释放语义:两个序列共享一块,一个释放后 refcount 从 2 变 1,块不回收
3. 把压力测试的分配概率从 0.6 改成 0.9,观察碎片率是否恶化(高负载更碎)
4. 用第 8 节的 per_tok 公式,计算 block_size=16/32/64 时「一块」的显存,画对比表

## 🔗 延伸阅读

- [PagedAttention (SOSP'23)](https://arxiv.org/abs/2309.06180)
- [vLLM block_pool 源码](https://github.com/vllm-project/vllm/blob/main/vllm/v1/core/block_pool.py)
- [vLLM V1 博客: Anatomy of vLLM](https://blog.vllm.ai/2025/09/05/anatomy-of-vllm.html)

---
> 🎉 恭喜完成本课!下一课见!